In [ ]:
import re
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.impute import SimpleImputer, KNNImputer


# ============================================================
# 0. Configuration
# ============================================================

def find_project_root():
    """
    Locate the project root from either a Python script or a notebook.

    Expected minimum project structure:
    Project_Root/
    ├─ Code/
    └─ Data/
       └─ SCB-Mesozoic-Granite.xls

    The Result folder will be created automatically if it does not exist.
    """
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (
            (path / "Code").exists()
            and (path / "Data").exists()
        ):
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code and Data folders."
    )


PROJECT_ROOT = find_project_root()

IN_PATH = PROJECT_ROOT / "Data" / "SCB-Mesozoic-Granite.xls"

OUT_DIR = PROJECT_ROOT / "Result" / "01_Foldwise_Preprocessing_and_Ratio_Features"
OUT_DIR.mkdir(parents=True, exist_ok=True)


TYPE_COL = "Type"

NON_NUM_COLS = {
    "No.",
    "Sample",
    "Type",
    "Reference"
}

TYPE_VALUE_MAP = {
    "A-type": "A",
    "S-type": "S",
    "I-type": "I",
    "A Type": "A",
    "S Type": "S",
    "I Type": "I",
    "A": "A",
    "S": "S",
    "I": "I"
}

MAX_MISSING_PER_ROW = 3

RANDOM_SEED = 42

# Standard five-fold CV: N_REPEATS = 1
# Repeated five-fold CV: for example, 5 repeats x 5 folds, set N_REPEATS = 5
N_SPLITS = 5
N_REPEATS = 1

# Imputation methods
IMPUTATION_METHODS = ["global_mean", "knn"]

# KNN parameters
KNN_N_NEIGHBORS = 5
KNN_WEIGHTS = "distance"

# Major-element columns
MAJOR_COLS = [
    "SiO2(wt%)",
    "TiO2",
    "Al2O3",
    "Fe2O3t",
    "MgO",
    "CaO",
    "Na2O",
    "K2O",
    "MnO",
    "P2O5"
]

# Trace-element and rare-earth-element columns
TRACE_COLS = [
    "Ga(ppm)",
    "Rb",
    "Sr",
    "Y",
    "Zr",
    "Nb",
    "Ba",
    "La",
    "Ce",
    "Pr",
    "Nd",
    "Sm",
    "Eu",
    "Gd",
    "Tb",
    "Dy",
    "Ho",
    "Er",
    "Tm",
    "Yb",
    "Lu",
    "Hf",
    "Ta",
    "Pb",
    "Th",
    "U",
    "Cs"
]


# ============================================================
# 1. Cleaning of special values
# ============================================================

def clean_censored_value(x):
    """
    Clean special records in geochemical datasets.

    Rules:
    1. Records with explicit numerical values, such as '<34', '>100',
       and '<0.01', are converted by retaining the numerical part.
       For example, '<0.01' -> 0.01 and '>100' -> 100.

    2. Records without recoverable numerical values, such as '<d.l.',
       '<dl', 'bdl', and 'below detection limit', are treated as missing.

    3. Records such as '> upper limit' and '>upper limit' are treated
       as missing.

    4. Blank entries and common missing-value tokens such as 'nan',
       'n.d.', 'na', and '--' are treated as missing.

    5. Other non-empty strings that cannot be converted to numbers
       are handled later by removing the corresponding sample rows.
    """
    if x is None:
        return np.nan

    s = str(x).strip()

    if s == "":
        return np.nan

    s_low = s.lower().replace(" ", "")

    missing_tokens = {
        "nan",
        "na",
        "n/a",
        "nd",
        "n.d.",
        "null",
        "none",
        "-",
        "--",
        "—"
    }

    if s_low in missing_tokens:
        return np.nan

    below_dl_tokens = {
        "<d.l.",
        "<d.l",
        "<dl",
        "<detectionlimit",
        "bdl",
        "<bdl",
        "belowdetectionlimit",
        "belowdl"
    }

    if s_low in below_dl_tokens:
        return np.nan

    upper_limit_tokens = {
        ">upperlimit",
        ">u.l.",
        ">ul",
        "aboveupperlimit"
    }

    if s_low in upper_limit_tokens:
        return np.nan

    # Remove commas in numerical strings, for example 1,234.5.
    s2 = s.replace(",", "")

    # Extract numerical values from records such as '<34', '>100',
    # '<0.01', and '>=100'.
    m = re.match(r"^[<>≤≥]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)$", s2)

    if m:
        return m.group(1)

    return s2


def is_missing_after_clean(x):
    """
    Check whether a cleaned value should be treated as missing.
    """
    if x is None:
        return True

    if pd.isna(x):
        return True

    s = str(x).strip().lower()

    return s in {
        "",
        "nan",
        "na",
        "n/a",
        "null",
        "none"
    }


# ============================================================
# 2. Basic cleaning before fold splitting
#    Note: This step performs only basic cleaning, not final imputation.
# ============================================================

def load_and_basic_clean(in_path):
    """
    Basic cleaning steps allowed before fold splitting:

    1. Read the dataset.
    2. Clean special values such as '<0.01', '>100', '<d.l.',
       and '> upper limit'.
    3. Remove sample rows containing non-numeric strings that still
       cannot be parsed after cleaning.
    4. Remove samples with more than MAX_MISSING_PER_ROW missing
       raw features.
    5. Retain NaN values without final imputation.

    This step does not use A/S/I labels and does not learn any
    imputation parameters.
    """
    df_raw = pd.read_excel(in_path, header=0)
    df = df_raw.copy()

    df.columns = [str(c).strip() for c in df.columns]

    if TYPE_COL not in df.columns:
        raise ValueError(
            f"Type column '{TYPE_COL}' was not found. "
            f"Current columns: {df.columns.tolist()}"
        )

    # Normalize A/S/I label strings from the new dataset while keeping
    # the downstream workflow unchanged.
    df[TYPE_COL] = (
        df[TYPE_COL]
        .astype(str)
        .str.strip()
        .replace(TYPE_VALUE_MAP)
    )

    num_cols = [c for c in df.columns if c not in NON_NUM_COLS]

    # Clean special values.
    for c in num_cols:
        df[c] = df[c].apply(clean_censored_value)

    # Identify non-empty strings that cannot be converted to numbers.
    df_num_tmp = df[num_cols].apply(pd.to_numeric, errors="coerce")

    garbled_mask = pd.DataFrame(
        False,
        index=df.index,
        columns=num_cols
    )

    for c in num_cols:
        orig = df[c]
        conv = df_num_tmp[c]

        garbled_mask[c] = (
            (~orig.apply(is_missing_after_clean))
            &
            (conv.isna())
        )

    rows_with_garbled = garbled_mask.any(axis=1)
    garbled_rows_count = int(rows_with_garbled.sum())

    garbled_examples = []

    if garbled_rows_count > 0:
        for idx in df.index[rows_with_garbled]:
            bad_cols = garbled_mask.columns[garbled_mask.loc[idx]].tolist()

            for bc in bad_cols:
                garbled_examples.append({
                    "row_index": idx,
                    "column": bc,
                    "value_after_cleaning": df.loc[idx, bc]
                })

    df = df.loc[~rows_with_garbled].reset_index(drop=True)

    # Convert all numeric columns to numerical dtype.
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Summarize feature-wise missingness before row filtering.
    original_missing_summary_raw = pd.DataFrame({
        "Feature": num_cols,
        "Original_missing_count": [
            int(df[c].isna().sum()) for c in num_cols
        ],
        "Original_missing_percent": [
            float(df[c].isna().mean() * 100) for c in num_cols
        ]
    }).sort_values(
        "Original_missing_percent",
        ascending=False
    )

    # Remove samples with more than MAX_MISSING_PER_ROW missing values.
    missing_count_per_row = df[num_cols].isna().sum(axis=1)
    rows_too_missing = missing_count_per_row > MAX_MISSING_PER_ROW
    too_missing_count = int(rows_too_missing.sum())

    df = df.loc[~rows_too_missing].reset_index(drop=True)

    # Recalculate missingness after row filtering.
    original_missing_summary_after_filter = pd.DataFrame({
        "Feature": num_cols,
        "Original_missing_count_after_row_filter": [
            int(df[c].isna().sum()) for c in num_cols
        ],
        "Original_missing_percent_after_row_filter": [
            float(df[c].isna().mean() * 100) for c in num_cols
        ]
    }).sort_values(
        "Original_missing_percent_after_row_filter",
        ascending=False
    )

    basic_summary = pd.DataFrame({
        "Item": [
            "Raw sample count",
            "Removed rows with unparseable strings",
            f"Removed rows with original missing values > {MAX_MISSING_PER_ROW}",
            "Final sample count after basic cleaning",
            "Numeric feature count"
        ],
        "Value": [
            len(df_raw),
            garbled_rows_count,
            too_missing_count,
            len(df),
            len(num_cols)
        ]
    })

    type_count = df[TYPE_COL].value_counts(dropna=False).reset_index()
    type_count.columns = [TYPE_COL, "Count"]

    garbled_examples_df = pd.DataFrame(garbled_examples)

    return {
        "df_clean": df,
        "num_cols": num_cols,
        "basic_summary": basic_summary,
        "type_count": type_count,
        "missing_raw": original_missing_summary_raw,
        "missing_after_filter": original_missing_summary_after_filter,
        "garbled_examples": garbled_examples_df
    }


# ============================================================
# 3. Fold-internal IQR outlier boundaries
# ============================================================

def fit_iqr_bounds_on_train(train_df, numeric_cols):
    """
    Calculate global feature-wise IQR outlier boundaries using only
    the training set.

    Type labels are not used.
    """
    records = []

    for c in numeric_cols:
        x = train_df[c].astype(float)
        x_valid = x.dropna()

        if len(x_valid) < 4:
            records.append({
                "Feature": c,
                "Q1": np.nan,
                "Q3": np.nan,
                "IQR": np.nan,
                "Lower_bound": np.nan,
                "Upper_bound": np.nan,
                "Note": "Valid values < 4, skipped"
            })
            continue

        q1 = x_valid.quantile(0.25)
        q3 = x_valid.quantile(0.75)
        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            records.append({
                "Feature": c,
                "Q1": q1,
                "Q3": q3,
                "IQR": iqr,
                "Lower_bound": np.nan,
                "Upper_bound": np.nan,
                "Note": "IQR is 0 or NaN, skipped"
            })
            continue

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        records.append({
            "Feature": c,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower_bound": lower,
            "Upper_bound": upper,
            "Note": "OK"
        })

    return pd.DataFrame(records)


def apply_iqr_bounds_mark_nan(df_part, numeric_cols, bounds_df):
    """
    Apply training-derived IQR boundaries to either the training or
    test set. Values outside the boundaries are marked as NaN and
    are later filled by the fold-internal imputer.
    """
    df_out = df_part.copy()
    records = []

    bounds_map = bounds_df.set_index("Feature").to_dict(orient="index")

    for c in numeric_cols:
        b = bounds_map[c]
        lower = b["Lower_bound"]
        upper = b["Upper_bound"]

        if pd.isna(lower) or pd.isna(upper):
            out_n = 0
        else:
            out_mask = (df_out[c] < lower) | (df_out[c] > upper)
            out_n = int(out_mask.sum())

            if out_n > 0:
                df_out.loc[out_mask, c] = np.nan

        records.append({
            "Feature": c,
            "Outlier_count_marked_as_NaN": out_n
        })

    return df_out, pd.DataFrame(records)


# ============================================================
# 4. Fold-internal imputation
# ============================================================

def foldwise_impute(train_df, test_df, numeric_cols, method):
    """
    Perform fold-internal imputation.

    method:
    1. global_mean:
       Fit SimpleImputer(strategy='mean') only on the training set,
       then transform both the training and test sets.

    2. knn:
       Calculate scaling parameters only from the training set.
       Fit KNNImputer only on the training set, then transform both
       the training and test sets.
    """
    train_out = train_df.copy()
    test_out = test_df.copy()

    if method == "global_mean":
        imputer = SimpleImputer(strategy="mean")

        train_arr = imputer.fit_transform(train_out[numeric_cols])
        test_arr = imputer.transform(test_out[numeric_cols])

        train_out[numeric_cols] = train_arr
        test_out[numeric_cols] = test_arr

        fill_info = pd.DataFrame({
            "Feature": numeric_cols,
            "Train_missing_filled": [
                int(train_df[c].isna().sum()) for c in numeric_cols
            ],
            "Test_missing_filled": [
                int(test_df[c].isna().sum()) for c in numeric_cols
            ],
            "Imputation_method": "global_mean",
            "Train_mean_used": imputer.statistics_
        })

    elif method == "knn":
        train_input = train_out[numeric_cols].astype(float)
        test_input = test_out[numeric_cols].astype(float)

        all_nan_cols = [
            c for c in numeric_cols
            if train_input[c].isna().all()
        ]

        if len(all_nan_cols) > 0:
            raise ValueError(
                f"The following features are entirely missing in the "
                f"training set and cannot be imputed by KNN: {all_nan_cols}"
            )

        # KNN is scale-sensitive, so scaling parameters are calculated
        # only from the training set.
        train_means = train_input.mean(skipna=True)
        train_stds = train_input.std(skipna=True).replace(0, 1.0)

        train_scaled = (train_input - train_means) / train_stds
        test_scaled = (test_input - train_means) / train_stds

        imputer = KNNImputer(
            n_neighbors=KNN_N_NEIGHBORS,
            weights=KNN_WEIGHTS
        )

        train_scaled_arr = imputer.fit_transform(train_scaled)
        test_scaled_arr = imputer.transform(test_scaled)

        train_filled_arr = (
            train_scaled_arr * train_stds.to_numpy()
            + train_means.to_numpy()
        )

        test_filled_arr = (
            test_scaled_arr * train_stds.to_numpy()
            + train_means.to_numpy()
        )

        train_out[numeric_cols] = train_filled_arr
        test_out[numeric_cols] = test_filled_arr

        fill_info = pd.DataFrame({
            "Feature": numeric_cols,
            "Train_missing_filled": [
                int(train_df[c].isna().sum()) for c in numeric_cols
            ],
            "Test_missing_filled": [
                int(test_df[c].isna().sum()) for c in numeric_cols
            ],
            "Imputation_method": "knn",
            "Train_mean_for_scaling": train_means.values,
            "Train_std_for_scaling": train_stds.values
        })

    else:
        raise ValueError(f"Unknown imputation method: {method}")

    return train_out, test_out, fill_info


# ============================================================
# 5. Fold-internal ratio-feature construction
# ============================================================

def ratio_with_zero_denominator_as_zero(numer, denom):
    """
    Construct a ratio feature.

    Current rules:
    - If the denominator is 0, the ratio is set to 0 to avoid inf.
    - If the numerator or denominator is NaN, the result is NaN.
    - inf and -inf are replaced by NaN.

    Note:
    If zero denominators become frequent in later checks, an alternative
    strategy is to set denominator-zero ratios to NaN and then perform
    another fold-internal imputation step.
    """
    numer_num = pd.to_numeric(numer, errors="coerce")
    denom_num = pd.to_numeric(denom, errors="coerce")

    zero_mask = denom_num == 0
    zero_count = int(zero_mask.sum())

    ratio = numer_num / denom_num
    ratio = ratio.replace([np.inf, -np.inf], np.nan)

    ratio[zero_mask] = 0.0

    return ratio, zero_count


def build_pairwise_ratios(df, cols, prefix):
    """
    Construct pairwise ratios from a given variable list.

    For example, if cols = [A, B, C], this function generates
    A/B, A/C, and B/C according to the predefined order.
    """
    missing_cols = [c for c in cols if c not in df.columns]

    if missing_cols:
        raise ValueError(
            f"The following columns for {prefix} were not found. "
            f"Please check the column names: {missing_cols}"
        )

    data_dict = {}
    zero_records = []

    for a, b in combinations(cols, 2):
        new_col = f"{prefix}{a}/{b}"

        ratio, zero_count = ratio_with_zero_denominator_as_zero(
            df[a],
            df[b]
        )

        data_dict[new_col] = ratio

        if zero_count > 0:
            zero_records.append({
                "Ratio_feature": new_col,
                "Denominator": b,
                "Zero_denominator_count": zero_count
            })

    ratio_df = pd.DataFrame(data_dict, index=df.index)
    zero_summary = pd.DataFrame(zero_records)

    return ratio_df, zero_summary


def add_ratio_features(df):
    """
    Add major-element ratios and trace-element/REE ratios.
    """
    major_ratio_df, major_zero = build_pairwise_ratios(
        df,
        MAJOR_COLS,
        prefix="R_Major_"
    )

    trace_ratio_df, trace_zero = build_pairwise_ratios(
        df,
        TRACE_COLS,
        prefix="R_Trace_"
    )

    df_with_ratios = pd.concat(
        [df, major_ratio_df, trace_ratio_df],
        axis=1
    )

    zero_summary = pd.concat(
        [
            major_zero.assign(Group="Major"),
            trace_zero.assign(Group="Trace_REE")
        ],
        ignore_index=True
    )

    return df_with_ratios, zero_summary


# ============================================================
# 6. Cross-validation splitter
# ============================================================

def get_cv_splitter():
    """
    Return the cross-validation splitter.
    """
    if N_REPEATS == 1:
        return StratifiedKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=RANDOM_SEED
        )

    return RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=RANDOM_SEED
    )


# ============================================================
# 7. Model feature-column checks
# ============================================================

def get_model_feature_cols(df):
    """
    Get model feature columns by excluding non-modeling columns such
    as No., Sample, and Type.
    """
    exclude_cols = set(NON_NUM_COLS)

    feature_cols = [
        c for c in df.columns
        if c not in exclude_cols
    ]

    return feature_cols


def check_no_nan_inf_in_features(df, feature_cols, name):
    """
    Check whether NaN or inf values remain in model features.

    Non-modeling columns such as Type and Sample are excluded.
    """
    arr = df[feature_cols].to_numpy(dtype=float)

    nan_total = int(np.isnan(arr).sum())
    inf_total = int(np.isinf(arr).sum())

    if nan_total > 0:
        raise ValueError(
            f"NaN values remain in the model features of {name}: {nan_total}"
        )

    if inf_total > 0:
        raise ValueError(
            f"Inf values remain in the model features of {name}: {inf_total}"
        )


# ============================================================
# 8. Main function
# ============================================================

def main():
    # --------------------------------------------------------
    # 8.1 Basic cleaning of the raw dataset
    # --------------------------------------------------------
    clean_result = load_and_basic_clean(IN_PATH)

    df_clean = clean_result["df_clean"]
    num_cols = clean_result["num_cols"]

    summary_clean_path = OUT_DIR / "basic_cleaning_summary.xlsx"

    with pd.ExcelWriter(summary_clean_path, engine="openpyxl") as writer:
        clean_result["basic_summary"].to_excel(
            writer,
            sheet_name="basic_summary",
            index=False
        )

        clean_result["type_count"].to_excel(
            writer,
            sheet_name="type_count_after_cleaning",
            index=False
        )

        clean_result["missing_raw"].to_excel(
            writer,
            sheet_name="missing_raw",
            index=False
        )

        clean_result["missing_after_filter"].to_excel(
            writer,
            sheet_name="missing_after_filter",
            index=False
        )

        if not clean_result["garbled_examples"].empty:
            clean_result["garbled_examples"].to_excel(
                writer,
                sheet_name="garbled_examples",
                index=False
            )
        else:
            pd.DataFrame({
                "Message": ["No unparseable strings after cleaning."]
            }).to_excel(
                writer,
                sheet_name="garbled_examples",
                index=False
            )

    print(f"Raw basic cleaning completed: {len(df_clean)} samples retained.")
    print(f"Basic cleaning summary saved to: {summary_clean_path}")

    # --------------------------------------------------------
    # 8.2 Construct cross-validation splits
    # --------------------------------------------------------
    X_dummy = df_clean[num_cols]
    y = df_clean[TYPE_COL].astype(str)

    cv = get_cv_splitter()

    fold_summary_records = []
    split_index_records = []

    # --------------------------------------------------------
    # 8.3 Fold-internal preprocessing for each imputation method
    # --------------------------------------------------------
    for method in IMPUTATION_METHODS:
        method_dir = OUT_DIR / method
        method_dir.mkdir(parents=True, exist_ok=True)

        for fold_id, (train_idx, test_idx) in enumerate(
            cv.split(X_dummy, y),
            start=1
        ):
            train_df_raw = (
                df_clean.iloc[train_idx]
                .copy()
                .reset_index(drop=True)
            )

            test_df_raw = (
                df_clean.iloc[test_idx]
                .copy()
                .reset_index(drop=True)
            )

            split_index_records.append({
                "Method": method,
                "Fold": fold_id,
                "Train_indices": ",".join(map(str, train_idx.tolist())),
                "Test_indices": ",".join(map(str, test_idx.tolist()))
            })

            # ------------------------------------------------
            # 1. Calculate IQR outlier boundaries using only
            #    the training set.
            # ------------------------------------------------
            iqr_bounds = fit_iqr_bounds_on_train(
                train_df_raw,
                num_cols
            )

            # ------------------------------------------------
            # 2. Apply training-derived IQR boundaries to both
            #    training and test sets.
            # ------------------------------------------------
            train_nan, train_outlier_summary = apply_iqr_bounds_mark_nan(
                train_df_raw,
                num_cols,
                iqr_bounds
            )

            test_nan, test_outlier_summary = apply_iqr_bounds_mark_nan(
                test_df_raw,
                num_cols,
                iqr_bounds
            )

            # ------------------------------------------------
            # 3. Fit the imputer only on the training set and
            #    transform both training and test sets.
            # ------------------------------------------------
            train_imp, test_imp, fill_info = foldwise_impute(
                train_nan,
                test_nan,
                num_cols,
                method=method
            )

            # ------------------------------------------------
            # 4. Add ratio features within each fold.
            # ------------------------------------------------
            train_ratio, train_zero_summary = add_ratio_features(train_imp)
            test_ratio, test_zero_summary = add_ratio_features(test_imp)

            # ------------------------------------------------
            # 5. Check whether NaN or inf values remain in
            #    model features.
            # ------------------------------------------------
            feature_cols = get_model_feature_cols(train_ratio)

            check_no_nan_inf_in_features(
                train_ratio,
                feature_cols,
                f"{method} Fold {fold_id} train"
            )

            check_no_nan_inf_in_features(
                test_ratio,
                feature_cols,
                f"{method} Fold {fold_id} test"
            )

            # ------------------------------------------------
            # 6. Save fold-wise train/test datasets with ratios.
            # ------------------------------------------------
            train_out_path = method_dir / f"fold_{fold_id:02d}_train_with_ratios.xlsx"
            test_out_path = method_dir / f"fold_{fold_id:02d}_test_with_ratios.xlsx"

            train_ratio.to_excel(train_out_path, index=False)
            test_ratio.to_excel(test_out_path, index=False)

            # ------------------------------------------------
            # 7. Save fold-wise preprocessing details.
            # ------------------------------------------------
            fold_stat_path = method_dir / f"fold_{fold_id:02d}_preprocessing_details.xlsx"

            with pd.ExcelWriter(fold_stat_path, engine="openpyxl") as writer:
                iqr_bounds.to_excel(
                    writer,
                    sheet_name="train_iqr_bounds",
                    index=False
                )

                train_outlier_summary.to_excel(
                    writer,
                    sheet_name="train_outliers",
                    index=False
                )

                test_outlier_summary.to_excel(
                    writer,
                    sheet_name="test_outliers",
                    index=False
                )

                fill_info.to_excel(
                    writer,
                    sheet_name="imputation_info",
                    index=False
                )

                if not train_zero_summary.empty:
                    train_zero_summary.to_excel(
                        writer,
                        sheet_name="train_ratio_zero_den",
                        index=False
                    )
                else:
                    pd.DataFrame({
                        "Message": ["No zero denominator in train ratios."]
                    }).to_excel(
                        writer,
                        sheet_name="train_ratio_zero_den",
                        index=False
                    )

                if not test_zero_summary.empty:
                    test_zero_summary.to_excel(
                        writer,
                        sheet_name="test_ratio_zero_den",
                        index=False
                    )
                else:
                    pd.DataFrame({
                        "Message": ["No zero denominator in test ratios."]
                    }).to_excel(
                        writer,
                        sheet_name="test_ratio_zero_den",
                        index=False
                    )

            fold_summary_records.append({
                "Method": method,
                "Fold": fold_id,

                "Train_sample_count": len(train_ratio),
                "Test_sample_count": len(test_ratio),

                "Train_class_A": int(
                    (train_ratio[TYPE_COL].astype(str) == "A").sum()
                ),
                "Train_class_S": int(
                    (train_ratio[TYPE_COL].astype(str) == "S").sum()
                ),
                "Train_class_I": int(
                    (train_ratio[TYPE_COL].astype(str) == "I").sum()
                ),

                "Test_class_A": int(
                    (test_ratio[TYPE_COL].astype(str) == "A").sum()
                ),
                "Test_class_S": int(
                    (test_ratio[TYPE_COL].astype(str) == "S").sum()
                ),
                "Test_class_I": int(
                    (test_ratio[TYPE_COL].astype(str) == "I").sum()
                ),

                "Train_outliers_marked_as_NaN": int(
                    train_outlier_summary[
                        "Outlier_count_marked_as_NaN"
                    ].sum()
                ),
                "Test_outliers_marked_as_NaN": int(
                    test_outlier_summary[
                        "Outlier_count_marked_as_NaN"
                    ].sum()
                ),

                "Train_values_filled": int(
                    fill_info["Train_missing_filled"].sum()
                ),
                "Test_values_filled": int(
                    fill_info["Test_missing_filled"].sum()
                ),

                "Feature_column_count_with_ratios": len(feature_cols),
                "Final_column_count_with_metadata": train_ratio.shape[1],

                "Train_file": str(train_out_path),
                "Test_file": str(test_out_path),
                "Detail_file": str(fold_stat_path)
            })

            print(
                f"{method} Fold {fold_id}: "
                f"train={len(train_ratio)}, test={len(test_ratio)}, "
                f"features={len(feature_cols)}"
            )

    # --------------------------------------------------------
    # 8.4 Save overall summary files.
    # --------------------------------------------------------
    fold_summary_df = pd.DataFrame(fold_summary_records)
    split_index_df = pd.DataFrame(split_index_records)

    summary_out_path = OUT_DIR / "foldwise_preprocessing_summary.xlsx"
    split_out_path = OUT_DIR / "cv_split_indices.xlsx"

    fold_summary_df.to_excel(summary_out_path, index=False)
    split_index_df.to_excel(split_out_path, index=False)

    print("\nAll tasks completed.")
    print(f"Output directory: {OUT_DIR}")
    print(f"Fold-wise preprocessing summary: {summary_out_path}")
    print(f"Cross-validation split indices: {split_out_path}")


# ============================================================
# 9. Program entry point
# ============================================================

if __name__ == "__main__":
    main()

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# 0. Path settings
# ============================================================

def find_project_root():
    if "__file__" in globals():
        start_dir = Path(__file__).resolve().parent
    else:
        start_dir = Path.cwd().resolve()

    for path in [start_dir] + list(start_dir.parents):
        if (path / "Code").exists() and (path / "Data").exists() and (path / "Result").exists():
            return path

    raise FileNotFoundError(
        "Project root was not found. Please make sure the project contains Code, Data, and Result folders."
    )

PROJECT_ROOT = find_project_root()


IN_PATH = PROJECT_ROOT / "Data" / "ASIGranite.xls"

OUT_DIR = PROJECT_ROOT / "Result" / "01_Missingness_and_Class_Distribution"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TYPE_COL = "Type"

NON_NUM_COLS = {
    "No.",
    "No",
    "Samp1e",
    "Sample",
    "Sample_ID",
    "SampleID",
    "ID",
    "Reference",
    "Type",
    "Type-1",
    "Type-2"
}

MAX_MISSING_PER_ROW = 3


# ============================================================
# 1. Global plotting style
# ============================================================

plt.rcParams["font.family"] = "Arial"
plt.rcParams["axes.titlesize"] = 22
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 18
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["xtick.labelsize"] = 14
plt.rcParams["ytick.labelsize"] = 14
plt.rcParams["legend.fontsize"] = 15
plt.rcParams["figure.titlesize"] = 22


# ============================================================
# 2. Special-value cleaning function
#    This function is consistent with the formal preprocessing script.
# ============================================================

def clean_censored_value(x):
    """
    Clean special records in geochemical datasets.

    Rules:
    1. Records with explicit numerical values, such as '<34', '>100',
       and '<0.01', are converted by retaining the numerical part.
       For example, '<0.01' -> 0.01 and '>100' -> 100.

    2. Records without recoverable numerical values, such as '<d.l.',
       '<dl', 'bdl', and 'below detection limit', are treated as missing.

    3. Records such as '> upper limit' and '>upper limit' are treated
       as missing.

    4. Blank entries and common missing-value tokens such as 'nan',
       'n.d.', 'na', and '--' are treated as missing.

    5. Other non-empty strings that cannot be converted to numbers
       are handled later by removing the corresponding sample rows.
    """
    if x is None:
        return np.nan

    s = str(x).strip()

    if s == "":
        return np.nan

    s_low = s.lower().replace(" ", "")

    missing_tokens = {
        "nan",
        "na",
        "n/a",
        "nd",
        "n.d.",
        "null",
        "none",
        "-",
        "--",
        "—"
    }

    if s_low in missing_tokens:
        return np.nan

    below_dl_tokens = {
        "<d.l.",
        "<d.l",
        "<dl",
        "<detectionlimit",
        "bdl",
        "<bdl",
        "belowdetectionlimit",
        "belowdl"
    }

    if s_low in below_dl_tokens:
        return np.nan

    upper_limit_tokens = {
        ">upperlimit",
        ">u.l.",
        ">ul",
        "aboveupperlimit"
    }

    if s_low in upper_limit_tokens:
        return np.nan

    # Remove commas in numerical strings, for example 1,234.5.
    s2 = s.replace(",", "")

    # Extract numerical values from records such as '<34', '>100',
    # '<0.01', and '>=100'.
    m = re.match(r"^[<>≤≥]\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)$", s2)

    if m:
        return m.group(1)

    return s2


def is_missing_after_clean(x):
    """
    Check whether a cleaned value should be treated as missing.
    """
    if x is None:
        return True

    if pd.isna(x):
        return True

    s = str(x).strip().lower()

    return s in {
        "",
        "nan",
        "na",
        "n/a",
        "null",
        "none"
    }


def normalize_type_label(x):
    """
    Standardize Type labels to avoid counting errors caused by labels
    such as A-type, A, a, and I-type.
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().upper()
    s = s.replace(" ", "")
    s = s.replace("_", "-")

    if s in {"A", "A-TYPE", "ATYPE"}:
        return "A"

    if s in {"S", "S-TYPE", "STYPE"}:
        return "S"

    if s in {"I", "I-TYPE", "ITYPE"}:
        return "I"

    return s


# ============================================================
# 3. Read the raw dataset
# ============================================================

df_raw = pd.read_excel(IN_PATH, header=0)
df = df_raw.copy()

df.columns = [str(c).strip() for c in df.columns]

if TYPE_COL not in df.columns:
    raise ValueError(
        f"Type column '{TYPE_COL}' was not found. "
        f"Current columns: {df.columns.tolist()}"
    )

num_cols = [c for c in df.columns if c not in NON_NUM_COLS]

if len(num_cols) == 0:
    raise ValueError(
        "No numeric feature columns were detected. "
        "Please check NON_NUM_COLS or the Excel header."
    )

print("=" * 80)
print("Raw dataset loaded successfully.")
print(f"Raw sample count: {len(df_raw)}")
print(f"Numeric feature count: {len(num_cols)}")
print("=" * 80)


# ============================================================
# 4. Clean special values
# ============================================================

for c in num_cols:
    df[c] = df[c].apply(clean_censored_value)

df_num_tmp = df[num_cols].apply(pd.to_numeric, errors="coerce")


# ============================================================
# 5. Identify and remove rows containing non-empty strings
#    that still cannot be parsed as numerical values
# ============================================================

garbled_mask = pd.DataFrame(False, index=df.index, columns=num_cols)

for c in num_cols:
    orig = df[c]
    conv = df_num_tmp[c]

    garbled_mask[c] = (
        (~orig.apply(is_missing_after_clean))
        &
        (conv.isna())
    )

rows_with_garbled = garbled_mask.any(axis=1)
garbled_rows_count = int(rows_with_garbled.sum())

garbled_examples = []

if garbled_rows_count > 0:
    for idx in df.index[rows_with_garbled]:
        bad_cols = garbled_mask.columns[garbled_mask.loc[idx]].tolist()

        for bc in bad_cols:
            garbled_examples.append({
                "row_index": idx,
                "column": bc,
                "value_after_cleaning": df.loc[idx, bc]
            })

df = df.loc[~rows_with_garbled].reset_index(drop=True)

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"Rows removed due to unparseable strings: {garbled_rows_count}")


# ============================================================
# 6. Summarize feature-wise missingness before removing samples
#    with excessive missing values
# ============================================================

missing_before_row_filter = pd.DataFrame({
    "Feature": num_cols,
    "Missing_count_before_row_filter": [
        int(df[c].isna().sum()) for c in num_cols
    ],
    "Missing_percent_before_row_filter": [
        float(df[c].isna().mean() * 100) for c in num_cols
    ],
    "Valid_count_before_row_filter": [
        int(df[c].notna().sum()) for c in num_cols
    ],
}).sort_values(
    "Missing_percent_before_row_filter",
    ascending=False
).reset_index(drop=True)

missing_before_only = missing_before_row_filter[
    missing_before_row_filter["Missing_count_before_row_filter"] > 0
].copy()


# ============================================================
# 7. Remove samples with more than three missing values
# ============================================================

missing_count_per_row = df[num_cols].isna().sum(axis=1)

rows_too_missing = missing_count_per_row > MAX_MISSING_PER_ROW
too_missing_count = int(rows_too_missing.sum())

df_clean = df.loc[~rows_too_missing].reset_index(drop=True)

print(
    f"Samples removed due to more than {MAX_MISSING_PER_ROW} "
    f"missing features: {too_missing_count}"
)
print(f"Sample count after basic cleaning: {len(df_clean)}")


# ============================================================
# 8. Summarize feature-wise missingness after removing samples
#    with excessive missing values
# ============================================================

missing_after_row_filter = pd.DataFrame({
    "Feature": num_cols,
    "Missing_count_after_row_filter": [
        int(df_clean[c].isna().sum()) for c in num_cols
    ],
    "Missing_percent_after_row_filter": [
        float(df_clean[c].isna().mean() * 100) for c in num_cols
    ],
    "Valid_count_after_row_filter": [
        int(df_clean[c].notna().sum()) for c in num_cols
    ],
}).sort_values(
    "Missing_percent_after_row_filter",
    ascending=False
).reset_index(drop=True)

# Retain only features with missing values.
missing_after_only = missing_after_row_filter[
    missing_after_row_filter["Missing_count_after_row_filter"] > 0
].copy()

print("\nNumber of features still containing missing values after row filtering:")
print(len(missing_after_only))

if len(missing_after_only) > 0:
    print("\nFeatures with missing values:")
    print(missing_after_only[[
        "Feature",
        "Missing_count_after_row_filter",
        "Missing_percent_after_row_filter"
    ]])
else:
    print("\nNo feature contains missing values after row filtering.")


# ============================================================
# 9. Summarize A/S/I class counts and proportions after basic cleaning
# ============================================================

df_clean[TYPE_COL] = df_clean[TYPE_COL].apply(normalize_type_label)

type_order = ["A", "S", "I"]

type_counts_raw = df_clean[TYPE_COL].value_counts(dropna=False)

type_counts = (
    df_clean[TYPE_COL]
    .value_counts()
    .reindex(type_order)
    .fillna(0)
    .astype(int)
)

type_summary = pd.DataFrame({
    "Type": type_counts.index,
    "Count": type_counts.values
})

total_type_count = int(type_summary["Count"].sum())

if total_type_count > 0:
    type_summary["Percent"] = type_summary["Count"] / total_type_count * 100
else:
    type_summary["Percent"] = 0.0

print("\nRaw Type counts:")
print(type_counts_raw)

print("\nClass distribution after basic cleaning:")
print(type_summary)


# ============================================================
# 10. Save Excel summary
# ============================================================

summary_excel_path = OUT_DIR / "missingness_and_cleaned_type_distribution.xlsx"

basic_summary = pd.DataFrame({
    "Item": [
        "Raw sample count",
        "Removed rows with unparseable strings",
        f"Removed rows with original missing values > {MAX_MISSING_PER_ROW}",
        "Final sample count after basic cleaning",
        "Numeric feature count",
        "Features with missing values before row filter",
        "Features with missing values after row filter"
    ],
    "Value": [
        len(df_raw),
        garbled_rows_count,
        too_missing_count,
        len(df_clean),
        len(num_cols),
        len(missing_before_only),
        len(missing_after_only)
    ]
})

garbled_examples_df = pd.DataFrame(garbled_examples)

with pd.ExcelWriter(summary_excel_path, engine="openpyxl") as writer:
    basic_summary.to_excel(
        writer,
        sheet_name="basic_summary",
        index=False
    )

    missing_before_row_filter.to_excel(
        writer,
        sheet_name="missing_before_all",
        index=False
    )

    missing_before_only.to_excel(
        writer,
        sheet_name="missing_before_only",
        index=False
    )

    missing_after_row_filter.to_excel(
        writer,
        sheet_name="missing_after_all",
        index=False
    )

    missing_after_only.to_excel(
        writer,
        sheet_name="missing_after_only",
        index=False
    )

    type_counts_raw.reset_index().to_excel(
        writer,
        sheet_name="raw_type_counts",
        index=False
    )

    type_summary.to_excel(
        writer,
        sheet_name="type_distribution",
        index=False
    )

    if len(garbled_examples_df) > 0:
        garbled_examples_df.to_excel(
            writer,
            sheet_name="garbled_examples",
            index=False
        )
    else:
        pd.DataFrame({
            "Message": ["No unparseable strings after cleaning."]
        }).to_excel(
            writer,
            sheet_name="garbled_examples",
            index=False
        )

print(f"\nExcel summary saved to: {summary_excel_path}")


# ============================================================
# 11. Figure 1: Feature-wise missing proportions
#    Only features with missing values are plotted.
#    X-axis = feature names
#    Y-axis = missing proportion
# ============================================================

missing_fig_path = OUT_DIR / "missing_features_percent_after_basic_cleaning.png"

if len(missing_after_only) > 0:
    plot_df = missing_after_only.sort_values(
        "Missing_percent_after_row_filter",
        ascending=False
    ).reset_index(drop=True)

    n_features = len(plot_df)

    fig_width = max(10, n_features * 0.65)
    fig_height = 7.5

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    bars = ax.bar(
        plot_df["Feature"],
        plot_df["Missing_percent_after_row_filter"],
        edgecolor="black",
        linewidth=1.2
    )

    ax.set_xlabel(
        "Geochemical variables with missing values",
        fontsize=19,
        fontweight="bold",
        labelpad=12
    )

    ax.set_ylabel(
        "Missing values before imputation (%)",
        fontsize=19,
        fontweight="bold",
        labelpad=12
    )

    ax.set_title(
        "Feature-wise Missing Proportions After Basic Cleaning",
        fontsize=23,
        fontweight="bold",
        pad=12
    )

    ax.tick_params(axis="x", labelsize=15, width=1.5)
    ax.tick_params(axis="y", labelsize=15, width=1.5)

    plt.xticks(rotation=45, ha="right", fontweight="bold")
    plt.yticks(fontweight="bold")

    max_value = float(plot_df["Missing_percent_after_row_filter"].max())

    if max_value <= 0:
        max_value = 1.0

    ax.set_ylim(0, max_value * 1.22)

    for bar, value in zip(bars, plot_df["Missing_percent_after_row_filter"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max_value * 0.025,
            f"{value:.2f}%",
            ha="center",
            va="bottom",
            fontsize=13,
            fontweight="bold",
            rotation=0
        )

    ax.grid(axis="y", linestyle="--", alpha=0.45)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(missing_fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Missing-proportion figure saved to: {missing_fig_path}")

else:
    no_missing_txt = OUT_DIR / "no_missing_features_after_basic_cleaning.txt"

    with open(no_missing_txt, "w", encoding="utf-8") as f:
        f.write(
            "After basic cleaning and row-level missingness filtering, "
            "no feature has missing values before imputation.\n"
        )

    print("No missing-feature plot was generated because no feature contains missing values.")
    print(f"Information file saved to: {no_missing_txt}")


# ============================================================
# 12. Figure 2: Pie chart of A/S/I sample proportions after basic cleaning
# ============================================================

pie_fig_path = OUT_DIR / "type_distribution_after_basic_cleaning_pie.png"

pie_df = type_summary[type_summary["Count"] > 0].copy()

if len(pie_df) > 0 and int(pie_df["Count"].sum()) > 0:
    fig, ax = plt.subplots(figsize=(7.2, 7.2))

    labels = [
        f"{row.Type}-type\nn={row.Count}\n{row.Percent:.1f}%"
        for row in pie_df.itertuples(index=False)
    ]

    wedges, texts = ax.pie(
        pie_df["Count"].values,
        labels=labels,
        startangle=90,
        counterclock=False,
        labeldistance=1.08,
        textprops={
            "fontsize": 16,
            "fontweight": "bold"
        },
        wedgeprops={
            "edgecolor": "black",
            "linewidth": 1.2
        }
    )

    ax.set_title(
        "A/S/I Type Distribution After Basic Cleaning",
        fontsize=23,
        fontweight="bold",
        pad=4
    )

    ax.axis("equal")

    # Adjust the spacing between the title and the figure.
    plt.subplots_adjust(top=0.91, bottom=0.04)

    plt.savefig(pie_fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Type-distribution pie chart saved to: {pie_fig_path}")

else:
    no_type_txt = OUT_DIR / "no_valid_type_distribution_for_pie.txt"

    with open(no_type_txt, "w", encoding="utf-8") as f:
        f.write(
            "No valid A/S/I type counts were found after basic cleaning. "
            "Please check the Type column labels.\n"
        )

    print("No pie chart was generated because no valid A/S/I type counts were found.")
    print(f"Information file saved to: {no_type_txt}")


# ============================================================
# 13. Save the basic-cleaned dataset
# ============================================================

cleaned_data_path = OUT_DIR / "basic_cleaned_data_before_imputation.xlsx"

df_clean.to_excel(cleaned_data_path, index=False)

print(f"Basic-cleaned dataset before imputation saved to: {cleaned_data_path}")

print("\nAll tasks completed.")
print(f"Output directory: {OUT_DIR}")